<a href="https://colab.research.google.com/github/labonig/CMSC-487-HW-4/blob/main/_downloads/4e865243430a47a00d551ca0579a6f6c/cifar10_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The code here is modified from the cifar10_tutorial.

In [1]:
%matplotlib inline

### 1. Load and normalize CIFAR10


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

In [3]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:02<00:00, 61.0MB/s]


2. Define a Convolutional Neural Network
========================================


In [4]:
import torch.nn as nn
import torch.nn.functional as F

In [5]:
# Question 0: Basic CNN
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

In [12]:
# Question 1: Deeper CNN architecture
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5, padding='same')
        self.conv2 = nn.Conv2d(6, 50, 5, padding='same')
        #self.conv3 = nn.Conv2d(16, 48, 5, padding='same')
        #self.conv4 = nn.Conv2d(48, 96, 5, padding='same')
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(50 * 8 * 8, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # shape: 3 x 32 x 32

        # first convolutional layer
        x = F.relu(self.conv1(x))

        x = self.pool(x)
        # shape: 3 x 16 x 16

        # second convolutional layer
        x = F.relu(self.conv2(x))
        #x = F.relu(self.conv3(x))
        #x = F.relu(self.conv4(x))

        x = self.pool(x)
        # shape: 96 x 8 x 8

        # flatten all dimensions except batch
        x = torch.flatten(x, 1)

        # fully connected layer
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

deeper_net = DeepNet()

3. Define a Loss function and optimizer
=======================================


In [7]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

4. Train the network
====================

In [8]:
def train_network(nn_name, num_epochs):
  for epoch in range(num_epochs):  # loop over the dataset multiple times

      running_loss = 0.0
      for i, data in enumerate(trainloader, 0):
          # get the inputs; data is a list of [inputs, labels]
          inputs, labels = data

          # zero the parameter gradients
          optimizer.zero_grad()

          # forward + backward + optimize
          outputs = nn_name(inputs)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          # print statistics
          running_loss += loss.item()
          if i % 2000 == 1999:    # print every 2000 mini-batches
              print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
              running_loss = 0.0

  print('Finished Training')


In [54]:
# Training the tutorial network
train_network(net, 2)

[1,  2000] loss: 2.230
[1,  4000] loss: 1.907
[1,  6000] loss: 1.717
[1,  8000] loss: 1.645
[1, 10000] loss: 1.529
[1, 12000] loss: 1.500
[2,  2000] loss: 1.419
[2,  4000] loss: 1.390
[2,  6000] loss: 1.368
[2,  8000] loss: 1.311
[2, 10000] loss: 1.294
[2, 12000] loss: 1.303
Finished Training


In [55]:
# save the trained model:
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

In [13]:
# Training the deeper model with convolutional blocks
train_network(deeper_net, 10)


[1,  2000] loss: 2.305
[1,  4000] loss: 2.305
[1,  6000] loss: 2.306
[1,  8000] loss: 2.305
[1, 10000] loss: 2.305
[1, 12000] loss: 2.305
[2,  2000] loss: 2.305
[2,  4000] loss: 2.304
[2,  6000] loss: 2.306
[2,  8000] loss: 2.306
[2, 10000] loss: 2.305
[2, 12000] loss: 2.304
[3,  2000] loss: 2.304
[3,  4000] loss: 2.306
[3,  6000] loss: 2.305
[3,  8000] loss: 2.305
[3, 10000] loss: 2.305
[3, 12000] loss: 2.306
[4,  2000] loss: 2.306


KeyboardInterrupt: 

In [ ]:
# save the trained model:
PATH2 = './cifar_deeper_net.pth'
torch.save(deeper_net.state_dict(), PATH2)

5. Test the network on the test data
====================================

In [57]:
dataiter = iter(testloader)
images, labels = next(dataiter)

In [58]:
# loading back the saved model:
net = Net()
net.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [59]:
def test_network(nn_name):
  correct = 0
  total = 0
  # since we're not training, we don't need to calculate the gradients for our outputs
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          # calculate outputs by running images through the network
          outputs = nn_name(images)
          # the class with the highest energy is what we choose as prediction
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

In [60]:
# Testing the tutorial network
test_network(net)

Accuracy of the network on the 10000 test images: 56 %


In [61]:
# Testing the deeper network
test_network(deeper_net)

KeyboardInterrupt: 

In [62]:
def print_accuracy(nn_name):
  # prepare to count predictions for each class
  correct_pred = {classname: 0 for classname in classes}
  total_pred = {classname: 0 for classname in classes}

  # again no gradients needed
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          outputs = nn_name(images)
          _, predictions = torch.max(outputs, 1)
          # collect the correct predictions for each class
          for label, prediction in zip(labels, predictions):
              if label == prediction:
                  correct_pred[classes[label]] += 1
              total_pred[classes[label]] += 1

  # print accuracy for each class
  for classname, correct_count in correct_pred.items():
      accuracy = 100 * float(correct_count) / total_pred[classname]
      print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

In [63]:
# print the per-class accuracy of the tutorial network
print_accuracy(net)

Accuracy for class: plane is 56.4 %
Accuracy for class: car   is 66.0 %
Accuracy for class: bird  is 40.3 %
Accuracy for class: cat   is 26.6 %
Accuracy for class: deer  is 39.3 %
Accuracy for class: dog   is 55.9 %
Accuracy for class: frog  is 72.1 %
Accuracy for class: horse is 62.5 %
Accuracy for class: ship  is 81.7 %
Accuracy for class: truck is 64.4 %


In [ ]:
# print the per-class accuracy of the deeper network
print_accuracy(deeper_net)

In [ ]:
del dataiter